:
HistGradientBoostingClassifier
(perte log_loss, donc probabiliste)

Le reste n’est pas un modèle, mais une couche de décision heuristique basée sur une analyse d’erreurs.

In [1]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score


In [2]:
def feature_engineering(df):
    df = df.copy()
    
    df['pages_per_age'] = df['total_pages_visited'] / (df['age'] + 0.1)
    df['interaction_age_pages'] = df['age'] * df['total_pages_visited']
    df['is_active'] = (df['total_pages_visited'] > 2).astype(int)
    
    return df


In [3]:
def fn_sniper_score(df, ref_df):
    score = np.zeros(len(df))
    
    q_pages_age = ref_df['pages_per_age'].quantile(0.75)
    q_inter_low = ref_df['interaction_age_pages'].quantile(0.25)
    
    score += (df['total_pages_visited'] <= 2).astype(int) * 0.4
    score += (df['pages_per_age'] > q_pages_age).astype(int) * 0.4
    score += (df['interaction_age_pages'] < q_inter_low).astype(int) * 0.2
    
    return score


In [4]:
train = pd.read_csv("conversion_data_train.csv")
test  = pd.read_csv("conversion_data_test.csv")

X = feature_engineering(train.drop("converted", axis=1))
y = train["converted"]
X_test = feature_engineering(test)


In [5]:
cat_cols = ['country', 'source']

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

cat_idx = [X.columns.get_loc(c) for c in cat_cols]


In [6]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_proba = np.zeros(len(X))
test_proba = np.zeros(len(X_test))

print(" Entraînement OOF...")

for fold, (tr, va) in enumerate(skf.split(X, y)):
    model = HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_iter=400,
        max_depth=8,
        l2_regularization=0.1,
        categorical_features=cat_idx,
        random_state=42 + fold
    )
    
    model.fit(X.iloc[tr], y.iloc[tr])
    
    oof_proba[va] = model.predict_proba(X.iloc[va])[:, 1]
    test_proba += model.predict_proba(X_test)[:, 1]
    
    print(f"  Fold {fold+1}/{N_FOLDS} terminé")

test_proba /= N_FOLDS


 Entraînement OOF...
  Fold 1/5 terminé
  Fold 2/5 terminé
  Fold 3/5 terminé
  Fold 4/5 terminé
  Fold 5/5 terminé


In [7]:
sniper_oof  = fn_sniper_score(X, X)
sniper_test = fn_sniper_score(X_test, X)

SNIPER_WEIGHT = 0.15

oof_final  = np.clip(oof_proba  + SNIPER_WEIGHT * sniper_oof,  0, 1)
test_final = np.clip(test_proba + SNIPER_WEIGHT * sniper_test, 0, 1)


In [8]:
best_f1 = 0
best_threshold = 0

thresholds = np.linspace(0.05, 0.6, 500)

for t in thresholds:
    preds = (oof_final >= t).astype(int)
    score = f1_score(y, preds)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\n RÉSULTATS OOF")
print(f"F1       : {best_f1:.5f}")
print(f"Seuil    : {best_threshold:.4f}")
print(f"ROC AUC  : {roc_auc_score(y, oof_final):.5f}")



 RÉSULTATS OOF
F1       : 0.76918
Seuil    : 0.4545
ROC AUC  : 0.96961


In [9]:
submission = pd.DataFrame({
    "converted": (test_final >= best_threshold).astype(int)
})

submission.to_csv("submission_FN_SNIPER.csv", index=False)

print("\n submission_FN_SNIPER.csv générée")
print(f"Conversions détectées : {submission['converted'].sum()}")



 submission_FN_SNIPER.csv générée
Conversions détectées : 922
